# 09_gold_setup
Creates the Gold layer in its own schema, workspace.gold: the business star schema (dimensions and three facts) and the audit mart (fact_pipeline_run). Safe to re-run.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
spark.sql("USE workspace.gold")
print("using", spark.sql("SELECT current_schema()").first()[0])

**Creating all the dimension tables based on our data**

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS dim_date (
  date_key INT,            -- yyyymmdd, e.g. 20261019
  full_date DATE,
  day_of_month INT,
  day_name STRING,
  day_of_week INT,         -- 1 = Monday
  is_weekend BOOLEAN,
  week_of_year INT,
  month_num INT,
  month_name STRING,
  quarter INT,
  year INT,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_game (
  game_key INT,            -- the Steam appid
  appid INT,
  game_name STRING,
  is_free BOOLEAN,
  first_tracked_date DATE,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_publisher (
  publisher_key BIGINT,    -- deterministic hash of the name, -1 = unknown
  publisher_name STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_developer (
  developer_key BIGINT,    -- deterministic hash of the name, -1 = unknown
  developer_name STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_price_tier (
  price_tier_key INT,
  tier_name STRING,
  min_price_usd DOUBLE,
  max_price_usd DOUBLE,
  sort_order INT,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_owners_band (
  owners_band_key INT,
  band_label STRING,       -- e.g. "1M to 2M"
  owners_min BIGINT,
  owners_max BIGINT,
  sort_order INT,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_change_type (
  change_type_key INT,
  change_type STRING,
  description STRING,
  load_timestamp TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS dim_pipeline_layer (
  layer_key INT,
  layer_name STRING,
  step_order INT,
  load_timestamp TIMESTAMP
) USING DELTA
""")

display(spark.sql("SHOW TABLES IN workspace.gold"))

**Creating our fact tables**

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS fact_game_daily_snapshot (
  date_key INT,
  game_key INT,
  publisher_key BIGINT,
  developer_key BIGINT,
  price_tier_key INT,
  owners_band_key INT,
  price_usd DOUBLE,
  initialprice_usd DOUBLE,
  discount_pct DOUBLE,
  is_on_sale BOOLEAN,
  positive INT,
  negative INT,
  total_reviews INT,
  approval_rate DOUBLE,
  ccu INT,
  owners_min BIGINT,
  owners_max BIGINT,
  owners_mid BIGINT,       -- midpoint of the owners range, for estimates
  load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key, game_key)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS fact_price_change (
  date_key INT,
  game_key INT,
  publisher_key BIGINT,
  developer_key BIGINT,
  change_type_key INT,
  old_price_usd DOUBLE,
  new_price_usd DOUBLE,
  price_change_usd DOUBLE, -- new minus old
  old_discount_pct DOUBLE,
  new_discount_pct DOUBLE,
  old_initialprice_usd DOUBLE,
  new_initialprice_usd DOUBLE,
  load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key, game_key)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS fact_game_listing (
  date_key INT,            -- day the game was first tracked
  game_key INT,
  publisher_key BIGINT,
  developer_key BIGINT,
  price_tier_key INT,
  owners_band_key INT,
  load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key)
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS fact_pipeline_run (
  run_id STRING,
  date_key INT,
  layer_key INT,
  load_type STRING,
  status STRING,
  file_processed STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  duration_seconds DOUBLE,
  rows_inserted BIGINT,
  rows_updated BIGINT,
  rows_skipped_unchanged BIGINT,
  rows_quarantined BIGINT,
  error_message STRING,
  load_timestamp TIMESTAMP
) USING DELTA
CLUSTER BY (date_key)
""")

display(spark.sql("SHOW TABLES IN workspace.gold"))

**Filling dimension tables that never depend of daily increment data**

In [0]:
from pyspark.sql import functions as F

def overwrite(rows, schema, table):
    (spark.createDataFrame(rows, schema)
         .withColumn("load_timestamp", F.current_timestamp())
         .write.mode("overwrite").saveAsTable(table))

# price tier is based on the list price (initialprice), so a temporary sale doesn't move a game to another tier
overwrite([(1, "Free", 0.0, 0.0, 1),
           (2, "Under $5", 0.01, 4.99, 2),
           (3, "$5 to $20", 5.0, 19.99, 3),
           (4, "$20 to $40", 20.0, 39.99, 4),
           (5, "Over $40", 40.0, None, 5)],
          "price_tier_key int, tier_name string, min_price_usd double, max_price_usd double, sort_order int",
          "dim_price_tier")

overwrite([(1, "Sale started", "Discount went from 0 to above 0"),
           (2, "Sale ended", "Discount went from above 0 to 0"),
           (3, "Discount changed", "Game stayed on sale with a different discount"),
           (4, "Price increase", "List price went up"),
           (5, "Price decrease", "List price went down")],
          "change_type_key int, change_type string, description string",
          "dim_change_type")

overwrite([(1, "Raw-to-Bronze", 1), (2, "Bronze-to-Silver", 2),
           (3, "Silver-to-Gold", 3), (4, "Pipeline-Runner", 4)],
          "layer_key int, layer_name string, step_order int",
          "dim_pipeline_layer")

# owners bands: taken from the ranges SteamSpy actually uses; key = upper bound in thousands (stable and sortable)
def short(n):
    for div, suf in ((1_000_000, "M"), (1_000, "k")):
        if n >= div:
            return f"{n // div}{suf}"
    return str(n)

bands = (spark.table("workspace.steam.silver_steam")
         .select("owners_min", "owners_max").where("owners_max IS NOT NULL")
         .distinct().orderBy("owners_max").collect())
overwrite([(int(b.owners_max // 1000), f"{short(b.owners_min)} to {short(b.owners_max)}",
            int(b.owners_min), int(b.owners_max), i + 1) for i, b in enumerate(bands)],
          "owners_band_key int, band_label string, owners_min bigint, owners_max bigint, sort_order int",
          "dim_owners_band")

# unknown members, so a missing publisher or developer still has a valid key
overwrite([(-1, "Unknown")], "publisher_key bigint, publisher_name string", "dim_publisher")
overwrite([(-1, "Unknown")], "developer_key bigint, developer_name string", "dim_developer")

for t in ["dim_price_tier", "dim_change_type", "dim_pipeline_layer", "dim_owners_band"]:
    display(spark.table(t).drop("load_timestamp"))